In [1]:
import os
from ratelimit import limits, sleep_and_retry
import requests
from urllib3.util import Retry
import sqlite3
import json
from pathlib import Path
import re

In [2]:
x = %pwd
PROJECT_ROOT = Path(x).resolve().parent
DATA_DIRECTORY = PROJECT_ROOT / 'data'

In [3]:
conn = sqlite3.connect(DATA_DIRECTORY / 'league_data.db')
cursor = conn.cursor()

In [4]:
rows = cursor.execute("""
    SELECT
        tables.name AS table_name,
        columns.name AS column_name
    FROM sqlite_schema AS tables
    JOIN pragma_table_info(tables.name) AS columns
    WHERE tables.type = 'table'
      AND tables.name NOT LIKE 'sqlite_%'
    ORDER BY tables.name, columns.cid
""").fetchall()

conn.commit()

for table_name, column_name in rows:
    print(table_name, column_name)

bans match_id
bans team_id
bans ban_1
bans ban_2
bans ban_3
bans ban_4
bans ban_5
match_queue match_id
match_queue status
matches match_id
matches team_id
matches champ_1
matches champ_2
matches champ_3
matches champ_4
matches champ_5
matches patch


In [5]:
EPOCH_TIME_AUGUST2026 = 1786924800
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10
NUM_GAMES_PER_PLAYER = 1
CURRENT_PATCH = '16.16.1' # Update this with the current patch version

platforms = ['OC1', 'JP1', 'KR', 'BR1', 'LA1', 'LA2', 'NA1', 'TR1', 'RU', 'EUN1', 'EUW1', 'ME1', 'SG2', 'TW2', 'VN2']
regions = {'OC1':'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea', 'JP1': 'asia', 'KR': 'asia', 'BR1': 'americas', 'LA1': 'americas', 'LA2': 'americas', 'NA1': 'americas', 'TR1': 'europe', 'RU': 'europe', 'EUN1': 'europe', 'EUW1': 'europe', 'ME1': 'europe'}

champion_names_url = 'https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json'
master_division_url = 'https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
matches_by_player_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count={count}'
match_data_from_matchid = 'https://{region}.api.riotgames.com/lol/match/v5/matches/{matchId}'
api_key = os.getenv("RIOT_API_KEY")

headers = {
    'X-Riot-Token': api_key
}

In [6]:
session = requests.Session()
retries = Retry(total=10,
                backoff_factor=2,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=retries))

In [7]:
@sleep_and_retry
@limits(calls=95, period=TWO_MINUTES)
@limits(calls=18, period=ONE_SECOND)
def call_api(url, headers=None):
    response = session.get(url, headers=headers)

    if response.status_code >= 400:
        print(f'Status: {response.status_code} Url: {url}')
        return None
    
    return response

In [ ]:
# currently up to br1
for platform in platforms[:3]:
    player_data = call_api(master_division_url.format(platform=platform), headers)

    data = player_data.json()['entries']
    player_id = [player['puuid'] for player in data] # collects all the player puuids from the master division

    for puuid in player_id:
        url = matches_by_player_url.format(region=regions[platform],
                                           puuid=puuid,
                                           start_time=EPOCH_TIME_AUGUST2026,
                                           queue=RANKED_SOLO,
                                           count=NUM_GAMES_PER_PLAYER)
        response = call_api(url, headers)
        
        if not response:
            continue

        for match in response.json():
            cursor.execute('INSERT OR IGNORE INTO match_queue (match_id) VALUES (?)', (match,))

        conn.commit()
        print(f'Added match {match} to queue')

Added match OC1_708730706 to queue
Added match OC1_708720596 to queue
Added match OC1_708737038 to queue
Added match OC1_708718851 to queue
Added match OC1_708684132 to queue
Added match OC1_708359967 to queue
Added match OC1_708359967 to queue
Added match OC1_708723219 to queue
Added match OC1_708700395 to queue
Added match OC1_708734690 to queue
Added match OC1_708734690 to queue
Added match OC1_708716711 to queue
Added match OC1_708511401 to queue
Added match OC1_708543336 to queue
Added match OC1_708265702 to queue
Added match OC1_708661441 to queue
Added match OC1_708688766 to queue
Added match OC1_708712045 to queue
Added match OC1_708731726 to queue
Added match OC1_708731726 to queue
Added match OC1_708731726 to queue
Added match OC1_708733815 to queue
Added match OC1_708733815 to queue
Added match OC1_708733842 to queue
Added match OC1_708716854 to queue
Added match OC1_708690240 to queue
Added match OC1_708705126 to queue
Added match OC1_708491945 to queue
Added match OC1_7087

KeyboardInterrupt: 

In [ ]:
while True:

    cursor.execute('SELECT match_id FROM match_queue WHERE status = "pending" LIMIT 1')
    row = cursor.fetchone()
    
    if not row:
        break

    match_id = row[0]
    platform = match_id.split('_')[0]
    url = match_data_from_matchid.format(region=regions[platform],
                                         matchId=match_id)
    
    response = call_api(url, headers)

    if not response:
        cursor.execute("UPDATE match_queue SET status = 'failed' WHERE match_id = (?)", (match_id,))
        continue
    
    players = response.json()['info']['participants']

    patch_version_long = response.json()['info']['gameVersion']
    patch_version_short = re.search('[0-9]+.[0-9]+.', patch_version_long).group()

    current_champs = [(player['championId'], player['teamId']) for player in players]

    # team_picks is a list with first element team_Id and next 5 elements champion names
    team1_picks = [current_champs[0][1]] + [champ[0] for champ in current_champs[:5]]
    team2_picks = [current_champs[5][1]] + [champ[0] for champ in current_champs[5:]]

    cursor.execute('''INSERT OR IGNORE INTO matches 
        (match_id, team_id, champ_1, champ_2, champ_3, champ_4, champ_5, patch)
        VALUES (?, ?, ?, ?, ?, ?, ?)''', [match_id] + team1_picks + [patch_version_short])

    cursor.execute('''INSERT OR IGNORE INTO matches 
        (match_id, team_id, champ_1, champ_2, champ_3, champ_4, champ_5, patch)
        VALUES (?, ?, ?, ?, ?, ?, ?)''', [match_id] + team2_picks + [patch_version_short])
    
    # store the bans for each team in a dictionary wity key teamId value [bans]
    team1_bans = [ban['championId'] for ban in response.json()['info']['teams'][0]['bans']]
    team2_bans = [ban['championId'] for ban in response.json()['info']['teams'][0]['bans']]
    team1_id = response.json()['info']['teams'][0]['teamId']
    team2_id = response.json()['info']['teams'][1]['teamId']

    cursor.execute('''INSERT OR IGNORE INTO bans 
        (match_id, team_id, ban_1, ban_2, ban_3, ban_4, ban_5)
        VALUES (?, ?, ?, ?, ?, ?, ?)''',[match_id] + [team1_id] + [team1_bans])

    cursor.execute('''INSERT OR IGNORE INTO bans 
        (match_id, team_id, ban_1, ban_2, ban_3, ban_4, ban_5)
        VALUES (?, ?, ?, ?, ?, ?, ?)''',[match_id] + [team2_id] + [team2_bans])

    cursor.execute('UPDATE match_queue SET status = "processed" WHERE match_id = (?)', (match_id,))
    conn.commit()

cursor.close()
conn.close()

In [ ]:
x = "16.16.804.9184"
y = re.search('[0-9]+.[0-9]+.', x).group()
print(y)

16.16.


In [ ]:
url = match_data_from_matchid.format(region='SEA',
                                         matchId='OC1_708334472')
response = call_api(url, headers)
bans = response.json()['info']['participants']

KeyError: 'participants'

In [ ]:
team1_bans = [ban['championId'] for ban in response.json()['info']['teams'][0]['bans']]
team2_bans = [ban['championId'] for ban in response.json()['info']['teams'][0]['bans']]
team1_id = response.json()['info']['teams'][0]['teamId']
team2_id = response.json()['info']['teams'][1]['teamId']
bans = {team1_id:team1_bans, team2_id:team2_bans}
print(bans)

{100: [104, 800, 105, 163, 29], 200: [104, 800, 105, 163, 29]}


In [ ]:
players = response.json()['info']['participants']
current_champs = [(player['championName'], player['teamId']) for player in players]
team1_picks = [current_champs[0][1]] + [champ[0] for champ in current_champs[:5]]
team2_picks = [current_champs[5][1]] + [champ[0] for champ in current_champs[5:]]
print(team2_picks)

[200, 'Zaahen', 'Viego', 'Yone', 'Syndra', 'Sona']


In [ ]:
# get champion names
all_champion_names = list(call_api(champion_names_url.format(version=CURRENT_PATCH)).json()['data'].keys())
with open(DATA_DIRECTORY / 'champ_names.json', 'w') as f:
    json.dump(all_champion_names, f)
session.close()